> 在线讲义：[第 20 章 · Blackwell：tcgen05、Tensor Memory、2-SM MMA 与块缩放低精度](https://wikieden.github.io/cuda-to-tensor-core/lessons/20-blackwell-tcgen05.html) · 本书《CUDA 从线程到 Tensor Core》

# 第 20 章 · Blackwell：tcgen05、Tensor Memory、2-SM MMA 与块缩放低精度

本章回答：Hopper 的 GEMM 代码为什么在 Blackwell 上直接跑不了？Tensor Memory（TMEM）是什么，为什么累加器不放寄存器了？"一个线程发起 MMA"意味着什么？两个 SM 怎么合作算一个 MMA？NVFP4、MXFP8 这些"块缩放"格式是怎么回事？

配套讲义：仓库里 `cuda-programming/notes/lessons/20-blackwell-tcgen05.md`（讲解、动画、练习都在讲义里）。对应原文：PTX §9.7.18 TensorCore 5th Generation（`materials/text/ptx-isa-9.4-focus-mma-wgmma-tcgen05-tma.txt` 第 12040–17350 行：§9.7.18.1 Tensor Memory 第 12040 行、§9.7.18.5 Issue Granularity 第 13135 行、§9.7.18.7 分配第 13604 行、§9.7.18.10 MMA 第 14425 行）、§9.7.16.3 Block Scaling（第 4764 行）；Colfax「Writing GEMM Kernels Using Tensor Memory For NVIDIA Blackwell GPUs」（`text/blog-colfax-blackwell-tmem-gemm.txt` 第 43–127 行）；gau-nernst「tcgen05 for dummies」（`text/blog-gau-nernst-tcgen05-for-dummies.txt`，2-SM MMA 第 918–991 行）；Blackwell tuning guide（`text/blackwell-tuning-guide.txt`）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU：代码用 NumPy 在 CPU 上模拟 GPU 的规则（线程编号、32 字节一箱、32 个 bank……）。

In [ ]:
import math

import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数
print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第二步 TMEM 大小 | 128 × 512 × 4 B | `tmem_bytes()` | 262144 字节 = 256 KB |
| 第二步 按列申请 | ≥ 32、2 的幂 | `tmem_alloc_cols(20)` / `tmem_alloc_cols(100)` | 32 / 128 列 |
| 第二步 每个 warp 管 1/4 | 32 个 lane | `warp_lanes(w)` | 0–31、32–63、64–95、96–127 |
| 第五步 FP4 只有 16 个值 | E2M1 | `E2M1` | 0, 0.5, 1, 1.5, 2, 3, 4, 6 再加正负号 |
| 第五步 块缩放 | $x \approx \text{scale} \times q$ | `block_quantize(x, 32, "ue8m0")` / `block_quantize(x, 16, "e4m3")` | 相对误差：一个 scale 0.250，MXFP4 0.161，NVFP4 0.090 |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**常量 `E2M1`**（来自 `ch20_tcgen05_tmem.py`）

In [ ]:
E2M1 = np.array([0, 0.5, 1, 1.5, 2, 3, 4, 6])

**函数 `round_e4m3`**（来自 `ch20_tcgen05_tmem.py`）：把正数舍入到 FP8 E4M3（3 位尾数、最大 448）：用来存 NVFP4 的缩放因子。

In [ ]:
def round_e4m3(v: np.ndarray) -> np.ndarray:
    """把正数舍入到 FP8 E4M3（3 位尾数、最大 448）：用来存 NVFP4 的缩放因子。"""
    e = np.maximum(np.floor(np.log2(v)), -6)
    q = 2.0 ** (e - 3)
    return np.minimum(np.round(v / q) * q, 448)

**函数 `round_ue8m0`**（来自 `ch20_tcgen05_tmem.py`）：UE8M0：只有指数、没有尾数，缩放因子只能是 2 的幂（向上取，保证不溢出）。

In [ ]:
def round_ue8m0(v: np.ndarray) -> np.ndarray:
    """UE8M0：只有指数、没有尾数，缩放因子只能是 2 的幂（向上取，保证不溢出）。"""
    return 2.0 ** np.ceil(np.log2(v))

**函数 `to_e2m1`**（来自 `ch20_tcgen05_tmem.py`）：取最近的 E2M1 值（带符号，超出 ±6 的截到 ±6）。

In [ ]:
def to_e2m1(y: np.ndarray) -> np.ndarray:
    """取最近的 E2M1 值（带符号，超出 ±6 的截到 ±6）。"""
    mag = np.abs(y)
    idx = np.abs(mag[..., None] - E2M1).argmin(axis=-1)
    return np.sign(y) * E2M1[idx]

**函数 `block_quantize`**（来自 `ch20_tcgen05_tmem.py`）：分块缩放：每 block 个元素共享一个 scale，x ≈ scale × q。返回反量化后的值（用来算误差）。

In [ ]:
def block_quantize(x: np.ndarray, block: int, scale_fmt: str) -> np.ndarray:
    """分块缩放：每 block 个元素共享一个 scale，x ≈ scale × q。返回反量化后的值（用来算误差）。"""
    xb = x.reshape(-1, block)
    amax = np.abs(xb).max(axis=1, keepdims=True)
    raw = np.where(amax > 0, amax / 6, 1.0)                 # 让块里最大的数落在 FP4 的最大值 6 附近
    scale = round_ue8m0(raw) if scale_fmt == "ue8m0" else round_e4m3(raw)
    return (to_e2m1(xb / scale) * scale).ravel()

**函数 `tmem_alloc_cols`**（来自 `ch20_tcgen05_tmem.py`）：tcgen05.alloc 以列为单位：至少 32 列、必须是 2 的幂。

In [ ]:
def tmem_alloc_cols(cols_needed: int) -> int:
    """tcgen05.alloc 以列为单位：至少 32 列、必须是 2 的幂。"""
    return max(32, 1 << math.ceil(math.log2(cols_needed)))

**常量 `TMEM_LANES`, `TMEM_COLS`, `CELL_BYTES`**（来自 `ch20_tcgen05_tmem.py`）

In [ ]:
TMEM_LANES, TMEM_COLS, CELL_BYTES = 128, 512, 4

**函数 `tmem_bytes`**（来自 `ch20_tcgen05_tmem.py`）：每个 CTA 的 TMEM：128 行 × 512 列 × 每格 32 位。

In [ ]:
def tmem_bytes() -> int:
    """每个 CTA 的 TMEM：128 行 × 512 列 × 每格 32 位。"""
    return TMEM_LANES * TMEM_COLS * CELL_BYTES

**函数 `warp_lanes`**（来自 `ch20_tcgen05_tmem.py`）：每个 warp 只能访问 TMEM 的 1/4：warp w 管第 32w .. 32w + 31 行（lane）。

In [ ]:
def warp_lanes(warp: int) -> range:
    """每个 warp 只能访问 TMEM 的 1/4：warp w 管第 32w .. 32w + 31 行（lane）。"""
    return range(32 * (warp % 4), 32 * (warp % 4) + 32)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第二步 TMEM")
print(f"  128 行 × 512 列 × 4 字节 = {tmem_bytes()} 字节 = {tmem_bytes() // 1024} KB / SM")
print("  申请 20 列 →", tmem_alloc_cols(20), "列；申请 100 列 →", tmem_alloc_cols(100), "列")
print("  4 个 warp 各管的 lane：", [(r.start, r.stop - 1) for r in (warp_lanes(w) for w in range(4))])

In [ ]:
print("\n第四步 2-SM MMA：单 SM 的 M 最大 128，CTA pair 拼起来 M =", 128 * 2, "，每个 SM 只放一半的 B")

In [ ]:
print("\n第五步 FP4（E2M1）")
print("  非负值：", E2M1, "；加上负号一共", 2 * len(E2M1), "个编码（含 ±0）")
rng = np.random.default_rng(0)
x = rng.standard_normal(4096) * np.exp(rng.standard_normal(4096))
one_scale = to_e2m1(x / (np.abs(x).max() / 6)) * (np.abs(x).max() / 6)
mx = block_quantize(x, 32, "ue8m0")
nv = block_quantize(x, 16, "e4m3")
err = lambda y: np.sqrt(np.mean((y - x) ** 2)) / np.sqrt(np.mean(x ** 2))
print(f"  整个张量一个 scale 的相对误差：{err(one_scale):.3f}")
print(f"  MXFP4（32 个一块、2 的幂 scale）：{err(mx):.3f}")
print(f"  NVFP4（16 个一块、E4M3 scale）  ：{err(nv):.3f}")

## 3. 代码里才看得到的

1. **误差数字取决于编的数据**：`x` 是"正态 × 对数正态"，故意让大小差得很多；换一种分布数字会变，但"块越小、scale 越细，误差越小"的顺序不变。
2. **UE8M0 向上取 2 的幂**：保证缩放后不超过 6；E4M3 缩放因子有 3 位尾数，能更贴近 amax / 6，这是 NVFP4 精度更好的另一半原因（另一半是块更小）。
3. **真实的 NVFP4 还有一个张量级的 FP32 缩放**（〔旧知〕），这里省略了，只模拟块级缩放。

## 4. 自己改一改

- 固定 scale 格式为 E4M3，把块大小从 16 换成 32、64，看误差怎么变。
- 把 `x` 换成没有离群值的 `rng.standard_normal(4096)`，看三种方案的差距是不是变小了。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> Blackwell 不认 wgmma，换成 tcgen05；
> 累加器搬进 TMEM：128 行 512 列，按列申请、用完归还，只能 ld 出来再加工；
> 一个线程发 MMA，搬运一个线程、计算一个线程，其余线程做收尾；
> 两个 SM 拼成一对，M 能到 256；
> 低精度靠分块缩放：MX 三十二个一块、二的幂；NVFP4 十六个一块、FP8 缩放，Tensor Core 自动乘回去。